# 06 · CEvNS acceptance study

            Replaces `Acceptance_analyst.ipynb`. Aggregates the per-batch
            ``acceptance_e<N>.npz`` files produced by ``scripts/run_acceptance.py`` and
            plots acceptance vs S2 area for each electron multiplicity.

In [ ]:
# Auto-discover the package even if the notebook is launched from outside the repo.
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 110

In [ ]:
from pathlib import Path
            ACC_DIR = Path('outputs/acceptance')
            files = sorted(ACC_DIR.glob('batch_*/acceptance_e*.npz'))
            assert files, 'Run scripts/run_acceptance.py first to populate outputs/acceptance/.'

## 1 – Aggregate

In [ ]:
from collections import defaultdict
            per_n = defaultdict(list)
            for fp in files:
                d = np.load(fp)
                per_n[int(d['e_num'][0])].append(d)

            for n in sorted(per_n):
                area = np.concatenate([d['area'] for d in per_n[n]])
                print(f'n_e={n}: {len(area)} events surviving the pattern cut')

## 2 – Acceptance vs area

In [ ]:
bins = np.linspace(0, 300, 31)
            fig, ax = plt.subplots(figsize=(7, 5))
            for n in sorted(per_n):
                area = np.concatenate([d['area'] for d in per_n[n]])
                hist, edges = np.histogram(area, bins=bins)
                ax.step(0.5*(edges[1:]+edges[:-1]), hist, label=f'n_e={n}')
            ax.set_xlabel('S2 area [pe]')
            ax.set_ylabel('events passing cut')
            ax.set_yscale('log')
            ax.legend()
            plt.show()

## 3 – Combined acceptance figure

            We typically combine the bins into a single acceptance-vs-area curve weighted
            by the assumed CEvNS spectrum.  Adapt the snippet below to your weighting.

In [ ]:
spectrum_path = 'data/e_spectrum_relics.npz'
            spec = np.load(spectrum_path)
            print('spectrum bins:', spec['CEvNS_bins'])
            print('spectrum counts:', spec['CEvNS'])